# MOSS-Image · T4 후보 스크리닝
[최신 Colab 노트북](https://github.com/HisameOgasahara/tmp_moss#readme)

**T4 GPU 선택 → 1~5 실행 → 결과표에서 통과 후보 선택.** 이미지 생성은 마지막 셀에서 실행합니다.


In [ ]:
#@title 1. T4 환경 확인
import sys, subprocess, json, os, time, shutil, hashlib, urllib.request
from pathlib import Path
from IPython.display import display, clear_output

ROOT = Path('/content/moss_t4')
ROOT.mkdir(parents=True, exist_ok=True)
gpu = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'
], text=True).strip()
if 'T4' not in gpu:
    raise RuntimeError(f'T4 GPU를 선택하세요. 현재 GPU: {gpu}')
print('GPU / VRAM / driver:', gpu)
print('RAM:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 2), 'GiB')
print('디스크 여유:', round(shutil.disk_usage(ROOT).free / 2**30, 2), 'GiB')
subprocess.run([sys.executable, '-c',
    'import torch; print("torch:", torch.__version__, "CUDA:", torch.version.cuda, "SM:", torch.cuda.get_device_capability())'], check=True)


In [ ]:
#@title 2. ComfyUI 설치
COMFY_REV = 'bc5f880dbdd43e426077f42ccec53bb577ffee9f'
MODEL_REV = '24147db865fcdc207270f844777587223fed8d6e'
FIXED_REV = 'ba6360149256790c5434e4a5b8ebf2c9899785b4'
SPECTRUM_REV = 'd3f7e243ea2db6cc3ff26f42b935b167f938b599'
COMFY = ROOT / 'ComfyUI'
VENV = ROOT / 'venv'
PY = str(VENV / 'bin/python')

def run(command, **kwargs):
    subprocess.run([str(x) for x in command], check=True, **kwargs)

def fetch(url):
    request = urllib.request.Request(url, headers={'User-Agent': 'moss-t4-smoke'})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read()

if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
run(['uv', 'venv', '--system-site-packages', '--python', sys.executable, VENV])
if not (COMFY / '.git').exists():
    run(['git', 'clone', '--quiet', 'https://github.com/Comfy-Org/ComfyUI.git', COMFY])
run(['git', '-C', COMFY, 'checkout', '--quiet', COMFY_REV])
# Colab의 기존 torch/CUDA 빌드를 유지합니다.
constraints = ROOT / 'torch-constraints.txt'
installed = subprocess.check_output([sys.executable, '-c',
    'import importlib.metadata as m; print(*(n+"=="+m.version(n) for n in ("torch","torchvision")), sep=chr(10))'], text=True)
constraints.write_text(installed)
run(['uv', 'pip', 'install', '--python', PY, '-c', constraints,
     '-r', COMFY / 'requirements.txt', 'huggingface_hub', 'psutil', 'pandas'])

source = json.loads(fetch('https://api.github.com/repos/HisameOgasahara/tmp_moss/commits/main'))['sha']
worker = ROOT / 'moss_smoke_worker.py'
worker.write_bytes(fetch(f'https://raw.githubusercontent.com/HisameOgasahara/tmp_moss/{source}/moss_smoke_worker.py'))
environment = {
    'source_revision': source, 'comfy_revision': COMFY_REV, 'model_revision': MODEL_REV,
    'fixed_revision': FIXED_REV, 'spectrum_revision': SPECTRUM_REV, 'gpu': gpu,
    'worker_sha256': hashlib.sha256(worker.read_bytes()).hexdigest(),
    'packages': subprocess.check_output([PY, '-m', 'pip', 'freeze'], text=True) if (VENV / 'bin/pip').exists()
        else subprocess.check_output(['uv', 'pip', 'freeze', '--python', PY], text=True),
}
(ROOT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print('설치 완료:', COMFY_REV[:12])


| ID | 본체 / 인코더 | 본체 연산 |
|---|---|---|
| N1 | INT8 / W4A8 | 자동 |
| N2 | INT8 / INT8 | 자동 |
| N3 | INT8 / INT8 | FP16 |
| N4 | FP16-fixed / INT8 | FP16 |

옵션: `lora8`, `spectrum`, `compile`, `pytorch_attention`, `tiled_vae`, `lowvram`, `fp8`.


In [ ]:
#@title 3. 후보 선택
BASE_IDS = 'N1,N2,N3,N4' #@param {type:'string'}
OPTION_BASE = 'N3' #@param ['N1','N2','N3','N4']
OPTIONS = 'lora8,spectrum,compile,pytorch_attention,tiled_vae,lowvram,fp8' #@param {type:'string'}
WIDTH = 1024 #@param {type:'integer'}
HEIGHT = 1024 #@param {type:'integer'}
PROMPT = 'A studio photograph of a red ceramic teapot beside a white card that reads "MOSS", soft window light, wooden table.' #@param {type:'string'}
SEED = 42 #@param {type:'integer'}
LOAD_TIMEOUT = 120 #@param {type:'integer'}
ENCODE_TIMEOUT = 60 #@param {type:'integer'}
FORWARD_TIMEOUT = 60 #@param {type:'integer'}
VAE_TIMEOUT = 30 #@param {type:'integer'}
COMPILE_TIMEOUT = 180 #@param {type:'integer'}
SPECTRUM_TIMEOUT = 180 #@param {type:'integer'}

import copy
if WIDTH <= 0 or HEIGHT <= 0 or WIDTH % 16 or HEIGHT % 16:
    raise ValueError('가로·세로는 양의 16배수여야 합니다.')
base = {
    'dit': 'moss_image_sft_int8_convrot.safetensors',
    'te': 'qwen3.5_9b_int8_convrot.safetensors', 'vae': 'flux2-vae_small.safetensors',
    'flags': [], 'option': 'none', 'weight_dtype': 'default',
    'comfy_root': str(COMFY), 'width': WIDTH, 'height': HEIGHT, 'prompt': PROMPT, 'seed': SEED,
}
registry = {key: dict(copy.deepcopy(base), id=key) for key in ['N1','N2','N3','N4']}
registry['N1']['te'] = 'qwen3.5_9b_w4a8_gptq.safetensors'
registry['N3']['flags'] = ['--fp16-unet']
registry['N4'].update(dit='moss_image_sft_fp16_fixed.safetensors', flags=['--fp16-unet'])
selected_ids = [value.strip() for value in BASE_IDS.split(',') if value.strip()]
unknown = set(selected_ids) - set(registry)
if unknown:
    raise ValueError(f'알 수 없는 후보: {unknown}')
candidates = [copy.deepcopy(registry[key]) for key in selected_ids]
valid_options = {'lora8','spectrum','compile','pytorch_attention','tiled_vae','lowvram','fp8'}
selected_options = list(dict.fromkeys(x.strip() for x in OPTIONS.split(',') if x.strip()))
if set(selected_options) - valid_options:
    raise ValueError(f'알 수 없는 옵션: {set(selected_options) - valid_options}')
for option in selected_options:
    item = dict(copy.deepcopy(registry[OPTION_BASE]), id=f'{OPTION_BASE}-{option}', option=option)
    if option == 'pytorch_attention':
        item['flags'].append('--use-pytorch-cross-attention')
    elif option == 'lowvram':
        item['flags'].extend(['--disable-dynamic-vram', '--lowvram'])
    elif option == 'fp8':
        item.update(dit='moss_image_sft_bf16.safetensors', weight_dtype='fp8_e4m3fn')
    elif option == 'lora8':
        item.update(lora='moss_sft_8step_lora.safetensors', lora_strength=1.0)
    elif option == 'spectrum':
        item.update(spectrum_path=str(ROOT / 'spectrum' / '__init__.py'), spectrum_calls=5,
                    spectrum_config={'mode':'manual','warmup_steps':2,'tail_actual_steps':0,
                                     'history_points':2,'chebyshev_degree':1})
    candidates.append(item)
if not candidates:
    raise ValueError('테스트 후보를 하나 이상 선택하세요.')
print('실행 후보:', ', '.join(item['id'] for item in candidates))


In [ ]:
#@title 4. 선택한 파일 다운로드
resources = {}
for candidate in candidates:
    for folder, key in [('diffusion_models','dit'), ('text_encoders','te'), ('vae','vae')]:
        name = candidate[key]
        fixed = name == 'moss_image_sft_fp16_fixed.safetensors'
        resources[(folder, name)] = {
            'repo': 'KBlueLeaf/MOSS-Image-FP16-fixed' if fixed else 'Comfy-Org/MOSS-Image',
            'revision': FIXED_REV if fixed else MODEL_REV,
            'filename': f'comfyui/{name}' if fixed else f'{folder}/{name}',
            'target': str(COMFY / 'models' / folder / name),
        }
if 'lora8' in selected_options:
    lora_repo = 'lodestones/MOSS-Image-SFT-8-steps-dmd'
    lora_rev = json.loads(fetch(f'https://huggingface.co/api/models/{lora_repo}'))['sha']
    resources[('loras', 'moss_sft_8step_lora.safetensors')] = {
        'repo': lora_repo, 'revision': lora_rev, 'filename': 'lora.safetensors',
        'target': str(COMFY / 'models/loras/moss_sft_8step_lora.safetensors'),
    }
manifest = ROOT / 'downloads.json'
manifest.write_text(json.dumps(list(resources.values()), indent=2), encoding='utf-8')
download_code = """
import json, os, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
for item in json.loads(Path(sys.argv[1]).read_text()):
    print('Download:', item['filename'], flush=True)
    source = hf_hub_download(item['repo'], item['filename'], revision=item['revision'], cache_dir=sys.argv[2])
    target = Path(item['target'])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink():
        target.unlink()
    if not target.exists():
        target.symlink_to(source)
"""
run([PY, '-c', download_code, manifest, ROOT / 'hf-cache'])
if 'spectrum' in selected_options:
    addon = ROOT / 'spectrum'
    addon.mkdir(exist_ok=True)
    for filename in ['__init__.py', 'nodes.py', 'er_sde_bridge.py']:
        addon.joinpath(filename).write_bytes(fetch(
            f'https://raw.githubusercontent.com/Rkkss/ComfyUI-Spectrum-MOSS/{SPECTRUM_REV}/{filename}'))
print('다운로드 완료')


In [ ]:
#@title 5. 스모크 실행·결과표
import csv, signal, zipfile
import pandas as pd

RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

def cgroup_oom_count():
    try:
        values = dict(line.split() for line in Path('/sys/fs/cgroup/memory.events').read_text().splitlines())
        return int(values.get('oom_kill', 0))
    except (OSError, ValueError):
        return 0

def read_state(path):
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return {}

def terminate_process(process):
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGKILL)
    process.wait()

def stage_limit(stage, item):
    if stage == 'generate':
        return item.get('generation_timeout', 1800)
    if stage.startswith('encode'):
        return ENCODE_TIMEOUT
    if stage.startswith('forward'):
        if item['option'] == 'compile':
            return COMPILE_TIMEOUT
        return SPECTRUM_TIMEOUT if item['option'] == 'spectrum' else FORWARD_TIMEOUT
    return VAE_TIMEOUT if stage == 'decode_vae' else LOAD_TIMEOUT

def run_one(item):
    tag = item['id'] + ('-generation' if item.get('mode') == 'generate' else '')
    state_path, result_path = RESULTS / f'{tag}.state.json', RESULTS / f'{tag}.json'
    task = dict(copy.deepcopy(item), state_path=str(state_path), result_path=str(result_path))
    task_path = RESULTS / f'{tag}.config.json'
    task_path.write_text(json.dumps(task, ensure_ascii=False, indent=2), encoding='utf-8')
    # A rerun starts with fresh state; its previous result stays only in the downloaded archive.
    for path in [state_path, result_path]:
        path.unlink(missing_ok=True)
    before_oom, started = cgroup_oom_count(), time.monotonic()
    peak_ram, peak_gpu, last_stage = 0.0, 0.0, None
    external_status = None
    import psutil
    with (RESULTS / f'{tag}.log').open('w') as log:
        process = subprocess.Popen([PY, '-u', worker, task_path], cwd=COMFY,
            stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            while process.poll() is None:
                state = read_state(state_path)
                stage = state.get('stage', 'import')
                if stage != last_stage:
                    print(f'{tag}: {stage}', flush=True)
                    last_stage = stage
                if time.monotonic() - state.get('stage_started', started) > stage_limit(stage, item):
                    external_status = 'TIMEOUT'
                    terminate_process(process)
                    break
                try:
                    parent = psutil.Process(process.pid)
                    family = [parent] + parent.children(recursive=True)
                    ram = 0
                    for child in family:
                        try:
                            ram += child.memory_info().rss
                        except psutil.Error:
                            pass
                    peak_ram = max(peak_ram, ram / 2**30)
                except psutil.Error:
                    pass
                try:
                    readings = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used',
                        '--format=csv,noheader,nounits'], text=True, timeout=3)
                    peak_gpu = max(peak_gpu, float(readings.splitlines()[0]) / 1024)
                except (subprocess.SubprocessError, ValueError):
                    pass
                time.sleep(0.5)
        finally:
            if process.poll() is None:
                terminate_process(process)
    result = read_state(result_path) or read_state(state_path) or {'candidate':item['id'], 'stage':'import'}
    if external_status:
        result.update(status=external_status, error='단계별 제한시간 초과')
    elif not result.get('status', '').startswith('PASS_') and process.returncode != 0:
        if cgroup_oom_count() > before_oom:
            result.update(status='HOST_OOM', error='cgroup oom_kill 증가')
        elif result.get('status') in (None, 'RUNNING'):
            result.update(status='PROCESS_EXIT', error=f'프로세스 종료 코드 {process.returncode}')
    elif process.returncode != 0:
        result.update(status='PROCESS_EXIT', error=f'프로세스 종료 코드 {process.returncode}')
    result.update(peak_host_rss_gib=round(peak_ram, 3), peak_gpu_used_gib=round(peak_gpu, 3),
                  elapsed_s=round(time.monotonic()-started, 3), config=task, exit_code=process.returncode)
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result

def show_results(results):
    rows = [{
        '후보':r['candidate'], '결과':r['status'], '단계':r.get('stage'),
        '초':r.get('elapsed_s'), 'VRAM GiB':r.get('peak_gpu_used_gib'),
        'RAM RSS GiB':r.get('peak_host_rss_gib'), '연산 dtype':r.get('dit_compute_dtype'),
        '오류':r.get('error',''),
    } for r in results]
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULTS / 'summary.csv', index=False, encoding='utf-8-sig')
    (RESULTS / 'summary.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
    display(frame)

results = []
for candidate in candidates:
    results.append(run_one(candidate))
    show_results(results)
passed = [r['candidate'] for r in results if r['status'] == 'PASS_SMOKE']
print('이미지 생성 후보:', ', '.join(passed) if passed else '없음')


In [ ]:
#@title 6. 결과·로그 다운로드
from google.colab import files
archive = ROOT / 'moss_t4_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in RESULTS.glob('*'):
        zipped.write(path, f'results/{path.name}')
    for path in [ROOT / 'environment.json', ROOT / 'downloads.json']:
        zipped.write(path, path.name)
files.download(str(archive))


In [ ]:
#@title 7. 선택한 통과 후보로 이미지 생성
RUN_GENERATION = False #@param {type:'boolean'}
CANDIDATE_ID = 'N3' #@param {type:'string'}
STEPS = 50 #@param {type:'integer'}
CFG = 3.0 #@param {type:'number'}
GENERATION_TIMEOUT = 1800 #@param {type:'integer'}

if RUN_GENERATION:
    matching = [r for r in results if r['candidate'] == CANDIDATE_ID and r['status'] == 'PASS_SMOKE']
    if not matching:
        raise ValueError('스모크를 통과한 후보 ID를 선택하세요.')
    task = copy.deepcopy(matching[0]['config'])
    task.update(mode='generate', steps=8 if task['option']=='lora8' else STEPS,
                cfg=1.0 if task['option']=='lora8' else CFG,
                generation_timeout=GENERATION_TIMEOUT,
                image_path=str(RESULTS / f'{CANDIDATE_ID}.png'))
    generated = run_one(task)
    if generated['status'] == 'PASS_GENERATION':
        from IPython.display import Image
        display(Image(filename=generated['image_path']))
    else:
        print(generated['status'], generated.get('stage'), generated.get('error'))
else:
    print('생성하려면 통과 후보 ID를 선택하고 RUN_GENERATION을 켜세요.')
